In [1]:
from pathlib import Path
import pandas as pd

# Répertoires à ignorer
IGNORED_DIRS = {
    ".git",
    ".venv",
    "venv",
    "env",
    "__pycache__",
    ".ipynb_checkpoints",
    "node_modules"
}

# Extensions intéressantes
INTERESTING_EXTENSIONS = {
    ".json",
    ".pkl",
    ".pickle",
    ".joblib",
    ".txt",
    ".model",
    ".pt",
    ".pth",
    ".npz",
    ".npy",
    ".parquet",
    ".csv"
}

# Mots-clés liés aux artefacts du PFE
KEYWORDS = [
    "lightgbm",
    "lgbm",
    "core",
    "model",
    "threshold",
    "feature",
    "prediction",
    "metric",
    "summary",
    "result",
    "graphsage",
    "gnn",
    "dynamic",
    "nomemory",
    "control",
    "graph",
    "processed",
    "artifact"
]

# Identifier automatiquement la racine probable du projet
cwd = Path.cwd().resolve()

root_candidates = [
    cwd,
    cwd.parent,
    cwd.parent.parent
]

def root_score(path):
    expected = ["data", "notebooks", "artifacts", "models", "results"]
    return sum((path / folder).exists() for folder in expected)

PROJECT_ROOT = max(root_candidates, key=root_score)

print("Répertoire courant :", cwd)
print("Racine détectée     :", PROJECT_ROOT)
print()

rows = []

for path in PROJECT_ROOT.rglob("*"):
    if not path.is_file():
        continue

    if any(part in IGNORED_DIRS for part in path.parts):
        continue

    suffix = path.suffix.lower()

    if suffix not in INTERESTING_EXTENSIONS:
        continue

    relative_path = str(path.relative_to(PROJECT_ROOT))
    lower_path = relative_path.lower()

    relevance = sum(keyword in lower_path for keyword in KEYWORDS)

    if relevance == 0:
        continue

    rows.append({
        "path": relative_path,
        "extension": suffix,
        "size_mb": round(path.stat().st_size / (1024 ** 2), 3),
        "relevance": relevance
    })

inventory = pd.DataFrame(rows)

if inventory.empty:
    print("Aucun artefact pertinent détecté.")
else:
    inventory = (
        inventory
        .sort_values(
            ["relevance", "path"],
            ascending=[False, True]
        )
        .reset_index(drop=True)
    )

    pd.set_option("display.max_rows", 200)
    pd.set_option("display.max_colwidth", 120)

    print(f"Artefacts pertinents détectés : {len(inventory)}")
    display(inventory.head(200))

Répertoire courant : C:\Users\bouza\OneDrive\Bureau\MobileMoney-Fraud-GNN\notebooks
Racine détectée     : C:\Users\bouza\OneDrive\Bureau\MobileMoney-Fraud-GNN

Artefacts pertinents détectés : 78


,path,extension,size_mb,relevance
0,artifacts\gnn\graphsage_core_edge_1layer_test_metrics.json,.json,0.000,6
1,artifacts\gnn\graphsage_core_edge_1layer_test_predictions.npz,.npz,2.648,6
2,artifacts\gnn\graphsage_core_edge_1layer_validation_predictions.npz,.npz,2.511,6
3,artifacts\gnn\graphsage_core_edge_2layer_validation_predictions.npz,.npz,2.175,6
4,artifacts\gnn\graphsage_core_edge_model_selection.json,.json,0.000,6
5,artifacts\dynamic_gnn\dynamic_core_edge_tbptt16_validation_metrics.json,.json,0.000,5
6,artifacts\dynamic_gnn\dynamic_core_edge_tbptt16_validation_predictions.npz,.npz,2.615,5
7,artifacts\dynamic_gnn\dynamic_core_edge_threshold.json,.json,0.000,5
8,artifacts\dynamic_gnn\dynamic_core_edge_validation_metrics.json,.json,0.000,5
9,artifacts\dynamic_gnn\dynamic_core_edge_validation_predictions.npz,.npz,2.924,5


In [2]:
import json

# Termes recherchés dans les métadonnées
SEARCH_TERMS = [
    "threshold",
    "feature",
    "features",
    "feature_set",
    "best_iteration",
    "pr_auc",
    "average_precision",
    "precision",
    "recall",
    "f1",
    "lightgbm",
    "lgbm",
    "validation",
    "test"
]

def search_json(obj, prefix=""):
    results = []

    if isinstance(obj, dict):
        for key, value in obj.items():
            current = f"{prefix}.{key}" if prefix else key

            if any(term in key.lower() for term in SEARCH_TERMS):
                if isinstance(value, (str, int, float, bool)) or value is None:
                    results.append((current, value))
                elif isinstance(value, list) and len(value) <= 50:
                    results.append((current, value))

            results.extend(search_json(value, current))

    elif isinstance(obj, list):
        for i, value in enumerate(obj):
            results.extend(search_json(value, f"{prefix}[{i}]"))

    return results


json_files = inventory[
    inventory["extension"].eq(".json")
]["path"].tolist()

print(f"JSON à inspecter : {len(json_files)}")

for relative_path in json_files:
    path = PROJECT_ROOT / relative_path

    try:
        with open(path, "r", encoding="utf-8") as f:
            content = json.load(f)

        matches = search_json(content)

        if matches:
            print("\n" + "=" * 80)
            print(relative_path)
            print("=" * 80)

            for key, value in matches[:80]:
                print(f"{key}: {value}")

    except Exception as e:
        print(f"\nImpossible de lire {relative_path}: {e}")

JSON à inspecter : 24

artifacts\gnn\graphsage_core_edge_1layer_test_metrics.json
pr_auc: 0.3527791922263579
threshold: 0.9998995065689087
precision: 0.9279411764705883
recall: 0.16475195822454308
f1: 0.2798226164079823
threshold_source: validation

artifacts\gnn\graphsage_core_edge_model_selection.json
validation_pr_auc: 0.24013615533229107
validation_roc_auc: 0.9623660083734987
threshold_selection: max_validation_f1
threshold: 0.9998995065689087
precision: 0.7062146892655368
recall: 0.19592476489028213
f1: 0.3067484662576687

artifacts\dynamic_gnn\dynamic_core_edge_tbptt16_validation_metrics.json
pr_auc: 0.39514428990244105
test_used: False

artifacts\dynamic_gnn\dynamic_core_edge_threshold.json
threshold: 0.9800333380699158
precision: 0.7309644670050761
recall: 0.45141065830721006
f1: 0.5581395348837209
pr_auc: 0.5473775841949017
test_used_for_selection: False

artifacts\dynamic_gnn\dynamic_core_edge_validation_metrics.json
pr_auc: 0.5473775841949017

artifacts\dynamic_gnn\no_memory

In [3]:
MODEL_EXTENSIONS = {
    ".joblib",
    ".pkl",
    ".pickle",
    ".model",
    ".txt"
}

model_candidates = inventory[
    inventory["extension"].isin(MODEL_EXTENSIONS)
].copy()

print("Candidats modèles détectés :")
display(model_candidates)

Candidats modèles détectés :


,path,extension,size_mb,relevance
36,models\lightgbm_core_baseline.txt,.txt,0.710,3
42,models\lightgbm_full_paysim_sensitive.txt,.txt,0.134,2


In [4]:
try:
    import pyarrow.parquet as pq

    parquet_files = inventory[
        inventory["extension"].eq(".parquet")
    ]["path"].tolist()

    print(f"Parquet détectés : {len(parquet_files)}")

    for relative_path in parquet_files:
        path = PROJECT_ROOT / relative_path

        try:
            parquet = pq.ParquetFile(path)

            print("\n" + "=" * 80)
            print(relative_path)
            print(f"Lignes     : {parquet.metadata.num_rows:,}")
            print(f"Row groups : {parquet.metadata.num_row_groups}")
            print(f"Colonnes   : {len(parquet.schema_arrow.names)}")
            print(parquet.schema_arrow.names)

        except Exception as e:
            print(f"\nErreur sur {relative_path}: {e}")

except ImportError:
    print("PyArrow n'est pas disponible dans cet environnement.")

Parquet détectés : 3

data\processed\paysim_graph_ready.parquet
Lignes     : 6,362,620
Row groups : 26
Colonnes   : 29
['nameOrig', 'nameDest', 'step', 'split', 'amount', 'log_amount', 'type', 'origin_balance_change', 'destination_balance_change', 'origin_zero_before', 'origin_zero_after', 'destination_zero_before', 'destination_zero_after', 'orig_prev_transactions', 'dest_prev_transactions', 'orig_prev_volume', 'dest_prev_volume', 'orig_time_since_last', 'dest_time_since_last', 'amount_to_oldbalance_ratio', 'origin_balance_residual', 'destination_balance_residual', 'origin_emptied', 'amount_equals_oldbalance', 'cycle_sin', 'cycle_cos', 'simulated_day', 'dest_active_last_24', 'isFraud']

data\processed\paysim_ml_features.parquet
Lignes     : 6,362,620
Row groups : 26
Colonnes   : 34
['step', 'split', 'amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest', 'log_amount', 'amount_to_dest_balance_ratio', 'origin_balance_change', 'destination_balance_change', 'origi

In [5]:
import json
import lightgbm as lgb

# Charger les métadonnées validées
metadata_path = PROJECT_ROOT / "results" / "ml_baseline_metadata.json"

with open(metadata_path, "r", encoding="utf-8") as f:
    ml_metadata = json.load(f)

core_metadata = ml_metadata["core_model"]

# Charger le modèle LightGBM Core
model_path = PROJECT_ROOT / "models" / "lightgbm_core_baseline.txt"

core_model = lgb.Booster(model_file=str(model_path))

# Informations du modèle sauvegardé
model_features = core_model.feature_name()
metadata_features = core_metadata["features"]

print("Modèle chargé :", model_path.exists())
print("Type :", type(core_model).__name__)
print()

print("Nombre de features modèle :", core_model.num_feature())
print("Nombre de features metadata :", len(metadata_features))
print()

print("Ordre identique :", model_features == metadata_features)
print()

print("Seuil Validation :", core_metadata["threshold"])
print("Best iteration metadata :", core_metadata["best_iteration"])
print("Best iteration modèle :", core_model.best_iteration)
print()

print("Features du modèle :")

for i, feature in enumerate(model_features, start=1):
    print(f"{i:02d}. {feature}")

print()
print("Catégories pandas sauvegardées :")
print(core_model.pandas_categorical)

Modèle chargé : True
Type : Booster

Nombre de features modèle : 22
Nombre de features metadata : 22

Ordre identique : True

Seuil Validation : 0.31370041201761084
Best iteration metadata : 209
Best iteration modèle : -1

Features du modèle :
01. amount
02. oldbalanceOrg
03. newbalanceOrig
04. oldbalanceDest
05. newbalanceDest
06. log_amount
07. amount_to_dest_balance_ratio
08. origin_balance_change
09. destination_balance_change
10. origin_zero_before
11. origin_zero_after
12. destination_zero_before
13. destination_zero_after
14. type
15. orig_prev_transactions
16. orig_prev_volume
17. orig_prev_mean_amount
18. orig_time_since_last
19. dest_prev_transactions
20. dest_prev_volume
21. dest_prev_mean_amount
22. dest_time_since_last

Catégories pandas sauvegardées :
[['CASH_IN', 'CASH_OUT', 'DEBIT', 'PAYMENT', 'TRANSFER']]


In [6]:
import pyarrow.parquet as pq
import pandas as pd
import numpy as np

ml_path = PROJECT_ROOT / "data" / "processed" / "paysim_ml_features.parquet"
graph_path = PROJECT_ROOT / "data" / "processed" / "paysim_graph_ready.parquet"

# Colonnes communes permettant de vérifier l'alignement
common_columns = [
    "step",
    "split",
    "amount",
    "type",
    "origin_balance_change",
    "destination_balance_change",
    "orig_prev_transactions",
    "dest_prev_transactions",
    "isFraud"
]

ml_parquet = pq.ParquetFile(ml_path)
graph_parquet = pq.ParquetFile(graph_path)

print("Nombre de row groups ML :", ml_parquet.num_row_groups)
print("Nombre de row groups Graph :", graph_parquet.num_row_groups)
print()

row_groups_to_test = [
    0,
    ml_parquet.num_row_groups // 2,
    ml_parquet.num_row_groups - 1
]

all_equal = True

for rg in row_groups_to_test:
    ml_sample = (
        ml_parquet
        .read_row_group(rg, columns=common_columns)
        .to_pandas()
    )

    graph_sample = (
        graph_parquet
        .read_row_group(rg, columns=common_columns)
        .to_pandas()
    )

    # Tester plusieurs positions dans le row group
    positions = sorted(set([
        0,
        len(ml_sample) // 2,
        len(ml_sample) - 1
    ]))

    ml_check = ml_sample.iloc[positions].reset_index(drop=True)
    graph_check = graph_sample.iloc[positions].reset_index(drop=True)

    equal = True

    for col in common_columns:
        if pd.api.types.is_numeric_dtype(ml_check[col]):
            col_equal = np.allclose(
                ml_check[col].to_numpy(),
                graph_check[col].to_numpy(),
                equal_nan=True
            )
        else:
            col_equal = (
                ml_check[col]
                .astype(str)
                .equals(graph_check[col].astype(str))
            )

        if not col_equal:
            equal = False
            print(f"Différence détectée : row group {rg}, colonne {col}")

    print(f"Row group {rg:02d} aligné : {equal}")

    all_equal &= equal

print()
print("Alignement échantillonné global :", all_equal)

Nombre de row groups ML : 26
Nombre de row groups Graph : 26

Row group 00 aligné : True
Row group 13 aligné : True
Row group 25 aligné : True

Alignement échantillonné global : True


In [7]:
# Vérifier les itérations réellement sauvegardées
print("Nombre d'arbres :", core_model.num_trees())
print("Itération courante :", core_model.current_iteration())
print("Best iteration metadata :", core_metadata["best_iteration"])

print()
print(
    "Nombre d'arbres cohérent avec le best iteration :",
    core_model.num_trees() == core_metadata["best_iteration"]
)

Nombre d'arbres : 209
Itération courante : 209
Best iteration metadata : 209

Nombre d'arbres cohérent avec le best iteration : True


In [8]:
import gc
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

ml_path = PROJECT_ROOT / "data" / "processed" / "paysim_ml_features.parquet"
graph_path = PROJECT_ROOT / "data" / "processed" / "paysim_graph_ready.parquet"

columns_to_check = [
    "step",
    "split",
    "amount",
    "type",
    "origin_balance_change",
    "destination_balance_change",
    "orig_prev_transactions",
    "dest_prev_transactions",
    "isFraud"
]

ml_parquet = pq.ParquetFile(ml_path)
graph_parquet = pq.ParquetFile(graph_path)

alignment_ok = True
checked_rows = 0

for rg in range(ml_parquet.num_row_groups):

    ml_rg = (
        ml_parquet
        .read_row_group(rg, columns=columns_to_check)
        .to_pandas()
    )

    graph_rg = (
        graph_parquet
        .read_row_group(rg, columns=columns_to_check)
        .to_pandas()
    )

    if len(ml_rg) != len(graph_rg):
        print(
            f"Row group {rg:02d} : taille différente "
            f"ML={len(ml_rg):,}, Graph={len(graph_rg):,}"
        )
        alignment_ok = False
        break

    rg_ok = True

    for col in columns_to_check:

        if pd.api.types.is_numeric_dtype(ml_rg[col]):

            col_ok = np.allclose(
                ml_rg[col].to_numpy(),
                graph_rg[col].to_numpy(),
                equal_nan=True
            )

        else:

            col_ok = (
                ml_rg[col]
                .astype(str)
                .reset_index(drop=True)
                .equals(
                    graph_rg[col]
                    .astype(str)
                    .reset_index(drop=True)
                )
            )

        if not col_ok:
            print(
                f"Différence détectée : "
                f"row group {rg:02d}, colonne {col}"
            )
            rg_ok = False
            alignment_ok = False
            break

    checked_rows += len(ml_rg)

    print(
        f"Row group {rg:02d} : "
        f"{len(ml_rg):,} lignes | aligné = {rg_ok}"
    )

    del ml_rg, graph_rg
    gc.collect()

    if not rg_ok:
        break

print()
print(f"Lignes vérifiées : {checked_rows:,}")
print("Alignement complet :", alignment_ok)

Row group 00 : 250,000 lignes | aligné = True
Row group 01 : 250,000 lignes | aligné = True
Row group 02 : 250,000 lignes | aligné = True
Row group 03 : 250,000 lignes | aligné = True
Row group 04 : 250,000 lignes | aligné = True
Row group 05 : 250,000 lignes | aligné = True
Row group 06 : 250,000 lignes | aligné = True
Row group 07 : 250,000 lignes | aligné = True
Row group 08 : 250,000 lignes | aligné = True
Row group 09 : 250,000 lignes | aligné = True
Row group 10 : 250,000 lignes | aligné = True
Row group 11 : 250,000 lignes | aligné = True
Row group 12 : 250,000 lignes | aligné = True
Row group 13 : 250,000 lignes | aligné = True
Row group 14 : 250,000 lignes | aligné = True
Row group 15 : 250,000 lignes | aligné = True
Row group 16 : 250,000 lignes | aligné = True
Row group 17 : 250,000 lignes | aligné = True
Row group 18 : 250,000 lignes | aligné = True
Row group 19 : 250,000 lignes | aligné = True
Row group 20 : 250,000 lignes | aligné = True
Row group 21 : 250,000 lignes | al

In [9]:
# Rechercher les artefacts liés au Machine Learning
patterns = [
    "lightgbm",
    "lgbm",
    "prediction",
    "predictions",
    "probability",
    "probabilities",
    "score",
    "scores",
    "ml_baseline"
]

mask = inventory["path"].str.lower().apply(
    lambda x: any(pattern in x for pattern in patterns)
)

ml_artifacts = (
    inventory[mask]
    .sort_values(["path"])
    .reset_index(drop=True)
)

print(f"Artefacts ML potentiellement utiles : {len(ml_artifacts)}")
display(ml_artifacts)

Artefacts ML potentiellement utiles : 13


,path,extension,size_mb,relevance
0,artifacts\dynamic_gnn\dynamic_core_edge_tbptt16_validation_predictions.npz,.npz,2.615,5
1,artifacts\dynamic_gnn\dynamic_core_edge_validation_predictions.npz,.npz,2.924,5
2,artifacts\dynamic_gnn\dynamic_gnn_final_test_predictions.npz,.npz,5.671,4
3,artifacts\dynamic_gnn\no_memory_control_validation_predictions.npz,.npz,2.697,5
4,artifacts\gnn\graphsage_core_edge_1layer_test_predictions.npz,.npz,2.648,6
5,artifacts\gnn\graphsage_core_edge_1layer_validation_predictions.npz,.npz,2.511,6
6,artifacts\gnn\graphsage_core_edge_2layer_validation_predictions.npz,.npz,2.175,6
7,models\lightgbm_core_baseline.txt,.txt,0.710,3
8,models\lightgbm_full_paysim_sensitive.txt,.txt,0.134,2
9,results\lightgbm_core_feature_importance.csv,.csv,0.001,4


In [10]:
import numpy as np
import pandas as pd

for relative_path in ml_artifacts["path"]:

    path = PROJECT_ROOT / relative_path
    suffix = path.suffix.lower()

    if suffix not in {".npz", ".npy", ".csv", ".parquet"}:
        continue

    print("\n" + "=" * 80)
    print(relative_path)
    print("=" * 80)

    try:

        if suffix == ".npz":

            with np.load(path, allow_pickle=False) as data:
                print("Clés :", data.files)

                for key in data.files:
                    arr = data[key]
                    print(
                        f"{key}: shape={arr.shape}, dtype={arr.dtype}"
                    )

        elif suffix == ".npy":

            arr = np.load(path, mmap_mode="r", allow_pickle=False)

            print(
                f"shape={arr.shape}, dtype={arr.dtype}"
            )

        elif suffix == ".csv":

            sample = pd.read_csv(path, nrows=5)

            print("Colonnes :", sample.columns.tolist())
            print("Aperçu :")
            display(sample)

        elif suffix == ".parquet":

            parquet = pq.ParquetFile(path)

            print(f"Lignes : {parquet.metadata.num_rows:,}")
            print("Colonnes :", parquet.schema_arrow.names)

    except Exception as e:
        print("Erreur :", e)


artifacts\dynamic_gnn\dynamic_core_edge_tbptt16_validation_predictions.npz
Clés : ['scores', 'labels', 'steps', 'src_known', 'dst_known']
scores: shape=(792406,), dtype=float32
labels: shape=(792406,), dtype=int8
steps: shape=(792406,), dtype=int16
src_known: shape=(792406,), dtype=bool
dst_known: shape=(792406,), dtype=bool

artifacts\dynamic_gnn\dynamic_core_edge_validation_predictions.npz
Clés : ['scores', 'labels', 'steps', 'type_idx', 'src_known', 'dst_known']
scores: shape=(792406,), dtype=float32
labels: shape=(792406,), dtype=int8
steps: shape=(792406,), dtype=int16
type_idx: shape=(792406,), dtype=int8
src_known: shape=(792406,), dtype=bool
dst_known: shape=(792406,), dtype=bool

artifacts\dynamic_gnn\dynamic_gnn_final_test_predictions.npz
Clés : ['labels', 'steps', 'type_idx', 'src_known', 'dst_known', 'dynamic_scores', 'control_scores']
labels: shape=(798827,), dtype=int8
steps: shape=(798827,), dtype=int16
type_idx: shape=(798827,), dtype=int8
src_known: shape=(798827,), d

,feature,gain,split,gain_pct
0,newbalanceDest,592960.735757,355,57.188486
1,oldbalanceDest,157647.265908,223,15.204394
2,log_amount,53954.498474,343,5.203677
3,oldbalanceOrg,49650.023456,968,4.788529
4,amount,44318.397500,1161,4.274317



results\lightgbm_core_test_performance_by_type.csv
Colonnes : ['type', 'frauds', 'tp', 'fn', 'recall', 'legitimate', 'fp', 'false_positive_rate', 'recall_pct', 'false_positive_rate_pct']
Aperçu :


,type,frauds,tp,fn,recall,legitimate,fp,false_positive_rate,recall_pct,false_positive_rate_pct
0,CASH_OUT,1915,1125,790,0.587467,269136,55,0.000204,58.746736,0.020436
1,TRANSFER,1915,1915,0,1.000000,68909,0,0.000000,100.000000,0.000000



results\lightgbm_full_feature_importance.csv
Colonnes : ['feature', 'gain', 'split', 'gain_pct', 'paysim_sensitive']
Aperçu :


,feature,gain,split,gain_pct,paysim_sensitive
0,amount_equals_oldbalance,2.620960e+06,1,99.664146,True
1,destination_balance_residual,2.670551e+03,114,0.101550,True
2,oldbalanceOrg,2.239936e+03,84,0.085175,False
3,origin_balance_change,1.605830e+03,59,0.061063,False
4,amount_to_dest_balance_ratio,5.910510e+02,145,0.022475,False


In [11]:
import gc
import numpy as np
import pandas as pd
import pyarrow.dataset as ds

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# Paramètres validés
feature_order = core_metadata["features"]
threshold = core_metadata["threshold"]
best_iteration = core_metadata["best_iteration"]

# Dataset ML
ml_dataset = ds.dataset(
    ml_path,
    format="parquet"
)

# Charger uniquement le Test et les colonnes nécessaires
columns = feature_order + [
    "step",
    "isFraud"
]

test_table = ml_dataset.to_table(
    columns=columns,
    filter=(
        (ds.field("step") >= 393)
        & (ds.field("step") <= 743)
    )
)

test_df = test_table.to_pandas()

print("Transactions Test :", f"{len(test_df):,}")
print("Fraudes Test      :", f"{test_df['isFraud'].sum():,}")
print(
    "Steps             :",
    test_df["step"].min(),
    "→",
    test_df["step"].max()
)

# Respecter les catégories du modèle
type_categories = core_model.pandas_categorical[0]

test_df["type"] = pd.Categorical(
    test_df["type"],
    categories=type_categories
)

# Préparer les variables
X_test = test_df[feature_order]
y_test = test_df["isFraud"].to_numpy()

# Calculer les probabilités
test_scores = core_model.predict(
    X_test,
    num_iteration=best_iteration
)

# Appliquer le seuil Validation
test_pred = (
    test_scores >= threshold
).astype(np.int8)

# Calculer les métriques
pr_auc = average_precision_score(
    y_test,
    test_scores
)

roc_auc = roc_auc_score(
    y_test,
    test_scores
)

precision = precision_score(
    y_test,
    test_pred
)

recall = recall_score(
    y_test,
    test_pred
)

f1 = f1_score(
    y_test,
    test_pred
)

tn, fp, fn, tp = confusion_matrix(
    y_test,
    test_pred
).ravel()

print()
print("Résultats reproduits")
print("--------------------")
print(f"PR-AUC    : {pr_auc:.12f}")
print(f"ROC-AUC   : {roc_auc:.12f}")
print(f"Precision : {precision:.12f}")
print(f"Recall    : {recall:.12f}")
print(f"F1        : {f1:.12f}")
print()
print(f"TP : {tp:,}")
print(f"FP : {fp:,}")
print(f"FN : {fn:,}")
print(f"TN : {tn:,}")

# Comparaison aux métadonnées
print()
print("Écarts avec les résultats validés")
print("---------------------------------")
print(
    "PR-AUC    :",
    abs(
        pr_auc
        - core_metadata["test_average_precision"]
    )
)

print(
    "ROC-AUC   :",
    abs(
        roc_auc
        - core_metadata["test_roc_auc"]
    )
)

print(
    "Precision :",
    abs(
        precision
        - core_metadata["test_precision"]
    )
)

print(
    "Recall    :",
    abs(
        recall
        - core_metadata["test_recall"]
    )
)

print(
    "F1        :",
    abs(
        f1
        - core_metadata["test_f1"]
    )
)

Transactions Test : 798,827
Fraudes Test      : 3,830
Steps             : 393 → 743

Résultats reproduits
--------------------
PR-AUC    : 0.979749048265
ROC-AUC   : 0.999890095321
Precision : 0.982229402262
Recall    : 0.793733681462
F1        : 0.877978339350

TP : 3,040
FP : 55
FN : 790
TN : 794,942

Écarts avec les résultats validés
---------------------------------
PR-AUC    : 0.0
ROC-AUC   : 0.0
Precision : 0.0
Recall    : 0.0
F1        : 0.0


## Préparation des artefacts applicatifs

L'audit a confirmé que le modèle LightGBM Core sauvegardé reproduit exactement les performances Test validées dans le notebook de Machine Learning.

Afin d'éviter de recalculer les probabilités de fraude à chaque lancement de l'application Streamlit, des artefacts légers dédiés à la couche applicative sont préparés.

Deux jeux de données sont construits :

- un fichier de prédictions contenant les scores du modèle sur l'ensemble Test ;
- un fichier d'alertes contenant uniquement les transactions dépassant le seuil sélectionné sur Validation.

Le `transaction_id` correspond à la position globale, indexée à partir de zéro, de la transaction dans les datasets processed. L'alignement complet entre `paysim_ml_features.parquet` et `paysim_graph_ready.parquet` ayant été vérifié sur les 6 362 620 transactions, cet identifiant permet de relier sans jointure coûteuse les informations ML et le contexte transactionnel du graphe.

Aucun nouvel entraînement, aucune optimisation de seuil et aucune modification du protocole expérimental ne sont réalisés.

In [12]:
from pathlib import Path
import gc
import json
import numpy as np
import pandas as pd
import pyarrow.dataset as ds
import pyarrow.parquet as pq

# Créer le répertoire applicatif
app_artifacts_dir = PROJECT_ROOT / "artifacts" / "app"
app_artifacts_dir.mkdir(parents=True, exist_ok=True)

predictions_path = (
    app_artifacts_dir
    / "lightgbm_core_test_predictions.parquet"
)

alerts_path = (
    app_artifacts_dir
    / "lightgbm_core_test_alerts.parquet"
)

manifest_path = (
    app_artifacts_dir
    / "app_manifest.json"
)

# Déterminer les identifiants globaux
total_transactions = pq.ParquetFile(
    ml_path
).metadata.num_rows

test_transactions = len(test_df)

test_start_id = (
    total_transactions
    - test_transactions
)

transaction_ids = np.arange(
    test_start_id,
    total_transactions,
    dtype=np.int64
)

print(
    "Premier transaction_id Test :",
    f"{transaction_ids[0]:,}"
)

print(
    "Dernier transaction_id Test  :",
    f"{transaction_ids[-1]:,}"
)

# Construire les prédictions applicatives
predictions_df = pd.DataFrame({
    "transaction_id": transaction_ids,
    "step": test_df["step"].to_numpy(
        dtype=np.int16
    ),
    "type": test_df["type"].astype(str).to_numpy(),
    "amount": test_df["amount"].to_numpy(),
    "fraud_probability": test_scores.astype(
        np.float32
    ),
    "prediction": test_pred.astype(
        np.int8
    ),
    "isFraud": y_test.astype(
        np.int8
    )
})

# Conserver les catégories PaySim
predictions_df["type"] = pd.Categorical(
    predictions_df["type"],
    categories=core_model.pandas_categorical[0]
)

print()
print(
    "Transactions dans predictions :",
    f"{len(predictions_df):,}"
)

print(
    "Alertes détectées :",
    f"{predictions_df['prediction'].sum():,}"
)

# Charger uniquement les colonnes nécessaires du graphe
graph_dataset = ds.dataset(
    graph_path,
    format="parquet"
)

graph_test_table = graph_dataset.to_table(
    columns=[
        "step",
        "nameOrig",
        "nameDest",
        "type",
        "amount",
        "isFraud"
    ],
    filter=(
        (ds.field("step") >= 393)
        & (ds.field("step") <= 743)
    )
)

graph_test = (
    graph_test_table
    .to_pandas()
    .reset_index(drop=True)
)

# Vérifier une dernière fois l'alignement Test
alignment_checks = {
    "nombre_lignes":
        len(graph_test)
        == len(predictions_df),

    "step":
        np.array_equal(
            graph_test["step"].to_numpy(),
            predictions_df["step"].to_numpy()
        ),

    "type":
        np.array_equal(
            graph_test["type"].astype(str).to_numpy(),
            predictions_df["type"].astype(str).to_numpy()
        ),

    "amount":
        np.allclose(
            graph_test["amount"].to_numpy(),
            predictions_df["amount"].to_numpy(),
            equal_nan=True
        ),

    "label":
        np.array_equal(
            graph_test["isFraud"].to_numpy(),
            predictions_df["isFraud"].to_numpy()
        )
}

print()
print("Alignement avant export :")

for key, value in alignment_checks.items():
    print(f"{key:15s} : {value}")

assert all(alignment_checks.values())

# Ajouter les identifiants globaux
graph_test.insert(
    0,
    "transaction_id",
    transaction_ids
)

# Extraire uniquement les alertes
alert_mask = (
    predictions_df["prediction"]
    .to_numpy(dtype=bool)
)

alerts_df = graph_test.loc[
    alert_mask,
    [
        "transaction_id",
        "step",
        "nameOrig",
        "nameDest",
        "type",
        "amount",
        "isFraud"
    ]
].copy()

alerts_df["fraud_probability"] = (
    predictions_df.loc[
        alert_mask,
        "fraud_probability"
    ].to_numpy()
)

alerts_df["prediction"] = np.int8(1)

# Réorganiser les colonnes
alerts_df = alerts_df[
    [
        "transaction_id",
        "step",
        "nameOrig",
        "nameDest",
        "type",
        "amount",
        "fraud_probability",
        "prediction",
        "isFraud"
    ]
]

# Sauvegarder les prédictions
predictions_df.to_parquet(
    predictions_path,
    index=False,
    engine="pyarrow",
    compression="zstd"
)

# Sauvegarder les alertes
alerts_df.to_parquet(
    alerts_path,
    index=False,
    engine="pyarrow",
    compression="zstd"
)

# Préparer le manifeste
manifest = {
    "application": {
        "purpose": (
            "Streamlit fraud monitoring "
            "and investigation demonstrator"
        )
    },

    "model": {
        "name": "LightGBM Core",
        "path": "models/lightgbm_core_baseline.txt",
        "feature_set": "Core",
        "n_features": 22,
        "best_iteration": int(
            core_metadata["best_iteration"]
        ),
        "threshold": float(
            core_metadata["threshold"]
        ),
        "threshold_source": "Validation Max F1"
    },

    "test_split": {
        "steps": [393, 743],
        "transactions": int(
            len(predictions_df)
        ),
        "frauds": int(
            predictions_df["isFraud"].sum()
        ),
        "alerts": int(
            predictions_df["prediction"].sum()
        )
    },

    "test_metrics": {
        "pr_auc": float(pr_auc),
        "roc_auc": float(roc_auc),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "tp": int(tp),
        "fp": int(fp),
        "fn": int(fn),
        "tn": int(tn)
    },

    "transaction_id": {
        "definition": (
            "Zero-based global row position "
            "in processed PaySim datasets"
        ),
        "first_test_id": int(
            transaction_ids[0]
        ),
        "last_test_id": int(
            transaction_ids[-1]
        ),
        "ml_graph_alignment_verified": True,
        "verified_rows": int(
            total_transactions
        )
    },

    "artifacts": {
        "predictions": (
            "artifacts/app/"
            "lightgbm_core_test_predictions.parquet"
        ),
        "alerts": (
            "artifacts/app/"
            "lightgbm_core_test_alerts.parquet"
        )
    }
}

with open(
    manifest_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        manifest,
        f,
        indent=2,
        ensure_ascii=False
    )

# Vérifier les fichiers générés
print()
print("Artefacts applicatifs")
print("---------------------")

for path in [
    predictions_path,
    alerts_path,
    manifest_path
]:
    print(
        f"{path.relative_to(PROJECT_ROOT)} "
        f"| {path.stat().st_size / 1024**2:.2f} MB"
    )

# Vérifier les lignes Parquet
predictions_parquet = pq.ParquetFile(
    predictions_path
)

alerts_parquet = pq.ParquetFile(
    alerts_path
)

print()
print(
    "Lignes predictions :",
    f"{predictions_parquet.metadata.num_rows:,}"
)

print(
    "Lignes alertes     :",
    f"{alerts_parquet.metadata.num_rows:,}"
)

print()
print("Préparation applicative terminée.")

# Libérer les objets temporaires lourds
del graph_test_table
del graph_test
gc.collect()

Premier transaction_id Test : 5,563,793
Dernier transaction_id Test  : 6,362,619

Transactions dans predictions : 798,827
Alertes détectées : 3,095

Alignement avant export :
nombre_lignes   : True
step            : True
type            : True
amount          : True
label           : True

Artefacts applicatifs
---------------------
artifacts\app\lightgbm_core_test_predictions.parquet | 8.82 MB
artifacts\app\lightgbm_core_test_alerts.parquet | 0.08 MB
artifacts\app\app_manifest.json | 0.00 MB

Lignes predictions : 798,827
Lignes alertes     : 3,095

Préparation applicative terminée.


122

In [13]:
import importlib
import sys

# Bibliothèques nécessaires à l'application
packages = [
    "streamlit",
    "pandas",
    "numpy",
    "pyarrow",
    "lightgbm",
    "plotly",
    "networkx"
]

print("Python :", sys.version)
print()

for package in packages:
    try:
        module = importlib.import_module(package)
        version = getattr(module, "__version__", "version non exposée")
        print(f"{package:12s} : OK | {version}")
    except ImportError:
        print(f"{package:12s} : MANQUANT")

Python : 3.13.7 (tags/v3.13.7:bcee1c3, Aug 14 2025, 14:15:11) [MSC v.1944 64 bit (AMD64)]

streamlit    : OK | 1.59.2
pandas       : OK | 3.0.3
numpy        : OK | 2.2.6
pyarrow      : OK | 24.0.0
lightgbm     : OK | 4.6.0
plotly       : OK | 6.9.0
networkx     : OK | 3.6.1


In [1]:
import numpy as np
import pandas as pd

from pathlib import Path


PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent


alerts_path = (
    PROJECT_ROOT
    / "artifacts"
    / "app"
    / "lightgbm_core_test_alerts.parquet"
)

dynamic_path = (
    PROJECT_ROOT
    / "artifacts"
    / "dynamic_gnn"
    / "dynamic_gnn_final_test_predictions.npz"
)

manifest_path = (
    PROJECT_ROOT
    / "artifacts"
    / "app"
    / "app_manifest.json"
)


import json

with open(
    manifest_path,
    "r",
    encoding="utf-8"
) as f:
    manifest = json.load(f)


alerts = pd.read_parquet(
    alerts_path
)

test_first_id = int(
    manifest["transaction_id"]["first_test_id"]
)


with np.load(
    dynamic_path,
    allow_pickle=False
) as data:
    src_known = data["src_known"]
    dst_known = data["dst_known"]


alerts_check = alerts.copy()

test_positions = (
    alerts_check["transaction_id"]
    .astype(np.int64)
    .to_numpy()
    - test_first_id
)


alerts_check["src_known"] = (
    src_known[test_positions]
)

alerts_check["dst_known"] = (
    dst_known[test_positions]
)


alerts_check["cold_start"] = np.select(
    [
        alerts_check["src_known"]
        & alerts_check["dst_known"],

        ~alerts_check["src_known"]
        & alerts_check["dst_known"],

        alerts_check["src_known"]
        & ~alerts_check["dst_known"]
    ],
    [
        "K→K",
        "N→K",
        "K→N"
    ],
    default="N→N"
)


print(
    alerts_check["cold_start"]
    .value_counts()
)

print()
print("Exemples N→K :")

display(
    alerts_check.loc[
        alerts_check["cold_start"].eq("N→K"),
        [
            "transaction_id",
            "step",
            "nameOrig",
            "nameDest",
            "type",
            "amount",
            "fraud_probability",
            "isFraud"
        ]
    ]
    .sort_values(
        "fraud_probability",
        ascending=False
    )
    .head(10)
)

cold_start
N→N    2256
N→K     839
Name: count, dtype: int64

Exemples N→K :


,transaction_id,step,nameOrig,nameDest,type,amount,fraud_probability,isFraud
245,5987933,411,C275137828,C989949797,CASH_OUT,1240717.75,0.999751,1
2139,6273187,635,C1928734777,C31164156,CASH_OUT,1363674.48,0.999741,1
2356,6284234,659,C780642333,C320130063,CASH_OUT,1423627.83,0.999719,1
2862,6361624,718,C1545616572,C1503189539,CASH_OUT,1293595.52,0.999708,1
290,5988262,415,C2137951962,C309573869,CASH_OUT,1675153.28,0.999696,1
429,6008427,434,C1140455895,C1640699182,CASH_OUT,1307156.17,0.999690,1
93,5786176,400,C579160579,C923916676,CASH_OUT,1191669.35,0.999685,1
467,6008703,438,C114220496,C1447772866,CASH_OUT,1648547.69,0.999681,1
1493,6168716,561,C602727752,C1182308451,CASH_OUT,1383818.98,0.999677,1
2779,6353931,708,C171779643,C808534536,CASH_OUT,1494146.93,0.999671,1


In [2]:
from pathlib import Path
import gc
import json
import sys
import time

import pandas as pd
import pyarrow.parquet as pq


# Détecter la racine du projet
cwd = Path.cwd().resolve()

if cwd.name == "notebooks":
    PROJECT_ROOT = cwd.parent
else:
    PROJECT_ROOT = cwd

APP_DIR = PROJECT_ROOT / "app"

if str(APP_DIR) not in sys.path:
    sys.path.insert(0, str(APP_DIR))


from utils.loaders import (
    ALERTS_PATH,
    GRAPH_DATA_PATH,
    MANIFEST_PATH,
    ML_DATA_PATH,
    MODEL_PATH,
    PREDICTIONS_PATH,
    load_alerts,
    load_core_model,
    load_manifest,
    load_predictions
)

from utils.investigation import (
    get_local_history,
    load_transaction_rows
)


# Mesurer la mémoire du processus si psutil est disponible
try:
    import psutil

    PROCESS = psutil.Process()

    def process_memory_mb():
        return PROCESS.memory_info().rss / 1024**2

    psutil_available = True

except ImportError:

    def process_memory_mb():
        return None

    psutil_available = False


def file_size_mb(path):
    # Calculer la taille d'un fichier
    return path.stat().st_size / 1024**2


def dataframe_memory_mb(df):
    # Calculer la mémoire réelle du DataFrame
    return (
        df.memory_usage(
            index=True,
            deep=True
        ).sum()
        / 1024**2
    )


def timed_load(name, loader):
    # Mesurer un chargement
    gc.collect()

    memory_before = process_memory_mb()

    start = time.perf_counter()
    result = loader()
    elapsed = time.perf_counter() - start

    memory_after = process_memory_mb()

    delta_memory = None

    if (
        memory_before is not None
        and memory_after is not None
    ):
        delta_memory = (
            memory_after
            - memory_before
        )

    print(f"{name}")
    print(f"  Temps : {elapsed:.4f} s")

    if delta_memory is not None:
        print(
            f"  Variation RSS : "
            f"{delta_memory:+.2f} MB"
        )

    return result, elapsed


print("=" * 70)
print("AUDIT PERFORMANCE / MÉMOIRE")
print("=" * 70)

print()
print("Racine :", PROJECT_ROOT)
print("Python :", sys.version.split()[0])
print("psutil :", psutil_available)


# Vérifier les tailles des artefacts applicatifs
print()
print("=" * 70)
print("1. TAILLE DES ARTEFACTS APPLICATIFS")
print("=" * 70)

paths = {
    "Prédictions Test": PREDICTIONS_PATH,
    "Alertes": ALERTS_PATH,
    "Manifest": MANIFEST_PATH,
    "LightGBM Core": MODEL_PATH
}

for name, path in paths.items():
    print(
        f"{name:20s} : "
        f"{file_size_mb(path):.3f} MB"
    )


# Vérifier les gros datasets sans les charger
print()
print("=" * 70)
print("2. DATASETS PROCESSED")
print("=" * 70)

for name, path in {
    "ML features": ML_DATA_PATH,
    "Graph ready": GRAPH_DATA_PATH
}.items():

    parquet = pq.ParquetFile(path)

    print()
    print(name)
    print(
        f"  Taille fichier : "
        f"{file_size_mb(path):.2f} MB"
    )
    print(
        f"  Lignes         : "
        f"{parquet.metadata.num_rows:,}"
    )
    print(
        f"  Row groups     : "
        f"{parquet.metadata.num_row_groups}"
    )


# Mesurer le manifeste
print()
print("=" * 70)
print("3. CHARGEMENTS STREAMLIT")
print("=" * 70)

manifest, manifest_time = timed_load(
    "Manifest",
    load_manifest
)


# Mesurer les prédictions
predictions, predictions_time = timed_load(
    "Prédictions Test",
    load_predictions
)

print(
    f"  Lignes : {len(predictions):,}"
)
print(
    f"  Mémoire DataFrame : "
    f"{dataframe_memory_mb(predictions):.2f} MB"
)


# Mesurer les alertes
alerts, alerts_time = timed_load(
    "Alertes",
    load_alerts
)

print(
    f"  Lignes : {len(alerts):,}"
)
print(
    f"  Mémoire DataFrame : "
    f"{dataframe_memory_mb(alerts):.2f} MB"
)


# Mesurer le modèle
model, model_time = timed_load(
    "LightGBM Core",
    load_core_model
)

print(
    f"  Features : {model.num_feature()}"
)
print(
    f"  Arbres   : {model.num_trees()}"
)


# Vérifier le cache Streamlit / fonctionnel
print()
print("=" * 70)
print("4. SECOND CHARGEMENT")
print("=" * 70)

_, predictions_cached_time = timed_load(
    "Prédictions Test - second appel",
    load_predictions
)

_, alerts_cached_time = timed_load(
    "Alertes - second appel",
    load_alerts
)

_, model_cached_time = timed_load(
    "LightGBM Core - second appel",
    load_core_model
)


# Tester une investigation réelle
print()
print("=" * 70)
print("5. INVESTIGATION CIBLÉE")
print("=" * 70)

transaction_id = 5_987_933

start = time.perf_counter()

ml_row, graph_row = load_transaction_rows(
    transaction_id
)

row_time = time.perf_counter() - start

print(
    f"Transaction : {transaction_id:,}"
)
print(
    f"Lecture ligne ciblée : "
    f"{row_time:.4f} s"
)

print(
    "Source      :",
    graph_row["nameOrig"]
)

print(
    "Destination :",
    graph_row["nameDest"]
)

print(
    "Step        :",
    graph_row["step"]
)


# Tester le contexte causal
start = time.perf_counter()

history = get_local_history(
    source=str(
        graph_row["nameOrig"]
    ),
    destination=str(
        graph_row["nameDest"]
    ),
    step=int(
        graph_row["step"]
    ),
    max_edges=12
)

history_time = (
    time.perf_counter()
    - start
)

print(
    f"Recherche historique : "
    f"{history_time:.4f} s"
)

print(
    f"Transactions historiques : "
    f"{len(history)}"
)


# Vérifier la cohérence applicative
print()
print("=" * 70)
print("6. COHÉRENCE")
print("=" * 70)

checks = {
    "predictions_798827":
        len(predictions) == 798_827,

    "alerts_3095":
        len(alerts) == 3_095,

    "model_22_features":
        model.num_feature() == 22,

    "model_209_trees":
        model.num_trees() == 209,

    "threshold_correct":
        abs(
            float(
                manifest["model"]["threshold"]
            )
            - 0.31370041201761084
        ) < 1e-15,

    "local_history_causal":
        (
            history.empty
            or (
                history["step"]
                < int(graph_row["step"])
            ).all()
        )
}

for name, status in checks.items():
    print(
        f"{name:25s} : {status}"
    )


# Rechercher les chargements potentiellement dangereux
print()
print("=" * 70)
print("7. AUDIT STATIQUE DE L'APPLICATION")
print("=" * 70)

suspicious_lines = []

for path in APP_DIR.rglob("*.py"):

    content = path.read_text(
        encoding="utf-8"
    )

    for line_number, line in enumerate(
        content.splitlines(),
        start=1
    ):

        stripped = line.strip()

        if (
            "paysim_ml_features.parquet"
            in stripped
            or "paysim_graph_ready.parquet"
            in stripped
        ):
            suspicious_lines.append(
                (
                    path.relative_to(
                        PROJECT_ROOT
                    ),
                    line_number,
                    stripped
                )
            )


if suspicious_lines:

    print(
        "Références aux gros Parquet détectées :"
    )

    for (
        path,
        line_number,
        line
    ) in suspicious_lines:

        print(
            f"  {path}:{line_number} "
            f"→ {line}"
        )

else:
    print(
        "Aucune référence directe aux gros "
        "Parquet dans les pages."
    )


print()
print("=" * 70)
print("8. RÉSUMÉ")
print("=" * 70)

print(
    f"Prédictions : "
    f"{predictions_time:.3f} s "
    f"| {dataframe_memory_mb(predictions):.2f} MB RAM"
)

print(
    f"Alertes     : "
    f"{alerts_time:.3f} s "
    f"| {dataframe_memory_mb(alerts):.2f} MB RAM"
)

print(
    f"Modèle      : "
    f"{model_time:.3f} s"
)

print(
    f"Ligne ciblée: "
    f"{row_time:.3f} s"
)

print(
    f"Historique  : "
    f"{history_time:.3f} s"
)

print()
print(
    "Tous les contrôles :",
    all(checks.values())
)

2026-09-02 07:56:36.018 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-09-02 07:56:36.022 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-09-02 07:56:36.024 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-09-02 07:56:36.027 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-09-02 07:56:36.031 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-09-02 07:56:36.033 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-09-02 07:56:36.037 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


AUDIT PERFORMANCE / MÉMOIRE

Racine : C:\Users\bouza\OneDrive\Bureau\MobileMoney-Fraud-GNN
Python : 3.13.7
psutil : True

1. TAILLE DES ARTEFACTS APPLICATIFS
Prédictions Test     : 8.820 MB
Alertes              : 0.078 MB
Manifest             : 0.001 MB
LightGBM Core        : 0.710 MB

2. DATASETS PROCESSED

ML features
  Taille fichier : 385.87 MB
  Lignes         : 6,362,620
  Row groups     : 26

Graph ready
  Taille fichier : 303.16 MB
  Lignes         : 6,362,620
  Row groups     : 26

3. CHARGEMENTS STREAMLIT


2026-09-02 07:56:36.487 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-09-02 07:56:36.489 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


Manifest
  Temps : 0.0048 s
  Variation RSS : +0.02 MB


2026-09-02 07:56:36.742 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-09-02 07:56:36.745 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-02 07:56:37.076 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-09-02 07:56:37.081 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


Prédictions Test
  Temps : 0.2093 s
  Variation RSS : +68.17 MB
  Lignes : 798,827
  Mémoire DataFrame : 19.05 MB
Alertes
  Temps : 0.0154 s
  Variation RSS : +0.39 MB
  Lignes : 3,095
  Mémoire DataFrame : 0.18 MB


2026-09-02 07:56:37.280 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


LightGBM Core
  Temps : 0.0269 s
  Variation RSS : +0.70 MB
  Features : 22
  Arbres   : 209

4. SECOND CHARGEMENT
Prédictions Test - second appel
  Temps : 0.0092 s
  Variation RSS : +19.07 MB
Alertes - second appel
  Temps : 0.0017 s
  Variation RSS : +0.29 MB
LightGBM Core - second appel
  Temps : 0.0001 s
  Variation RSS : +0.00 MB

5. INVESTIGATION CIBLÉE
Transaction : 5,987,933
Lecture ligne ciblée : 0.1783 s
Source      : C275137828
Destination : C989949797
Step        : 411
Recherche historique : 1.9806 s
Transactions historiques : 4

6. COHÉRENCE
predictions_798827        : True
alerts_3095               : True
model_22_features         : True
model_209_trees           : True
threshold_correct         : True
local_history_causal      : True

7. AUDIT STATIQUE DE L'APPLICATION
Références aux gros Parquet détectées :
  app\utils\loaders.py:37 → / "paysim_ml_features.parquet"
  app\utils\loaders.py:44 → / "paysim_graph_ready.parquet"

8. RÉSUMÉ
Prédictions : 0.209 s | 19.05 MB RA